Proposition #1

Return the union of all orders that include IT joke mugs (of hardware) and Developer joke mugs. This is to see all of the orders involving those types of items. (use Fact.Order)

In [ ]:
USE WideWorldImportersDW

-- get info on the orders of both types of mugs below
SELECT [WWI Order ID], [Customer Key], [Order Date Key], [Description], [Quantity], [Total Including Tax]
FROM Fact.[Order]
WHERE Description LIKE N'IT joke mug - hardware:%'

UNION

SELECT [WWI Order ID], [Customer Key], [Order Date Key], [Description], [Quantity], [Total Including Tax]
FROM Fact.[Order]
WHERE Description LIKE N'Developer joke mug%'

Proposition #2

Find the intersection of all of the customer keys of customers that made orders. This is to see how many customers have made orders. (use Dimension.Employee and Fact.Order)

In [ ]:
USE WideWorldImportersDW

-- get the customer keys of all customers that made orders
SELECT [Customer Key]
FROM Dimension.Customer

INTERSECT

SELECT [Customer Key]
FROM Fact.[Order]

Proposition #3

Find all customers whose IDs don't match with an employee's ID, and then return info about each remaining customer. This is to see how many more customers there are than employees in WWI. (use Dimension.Customer and Dimension.Employee)

In [ ]:
USE WideWorldImportersDW;

WITH MoreCust as        -- get all customers that don't have a matching ID to an employee's ID
(
    SELECT [WWI Customer ID]
    FROM Dimension.Customer

    EXCEPT

    SELECT [WWI Employee ID]
    FROM Dimension.Employee
)

SELECT D.[WWI Customer ID] as 'CustId', 
       [Customer], 
       [Primary Contact], 
       [Postal Code]
FROM MoreCust, Dimension.Customer as D
WHERE MoreCust.[WWI Customer ID] = D.[WWI Customer ID];     -- make sure to keep only customer IDs that don't match to an employee's ID

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Proposition #4</span>

<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Find all customers' orders that were only made in January 2016. This is to get the info of all customers and their orders that were made during a specific time. (use Dimension.Customer and Fact.Order)</span>

In [ ]:
USE WideWorldImportersDW;

WITH CustsOrdered as(       -- get all customers that have made orders
    SELECT [Customer Key]
    FROM Dimension.Customer

    INTERSECT

    SELECT [Customer Key]
    FROM Fact.[Order]
)


SELECT Customers.[WWI Customer ID], 
       Customers.[Customer], 
       Orders.[Order Date Key], 
       Orders.[Description], 
       Orders.[Quantity], 
       Orders.[Total Including Tax]

FROM CustsOrdered, 
     Dimension.[Customer] as Customers, 
     Fact.[Order] as Orders

WHERE '2016-01-01' <= Orders.[Order Date Key] AND                   -- get all orders in January 2016
      Orders.[Order Date Key] <= '2016-01-31' AND
      CustsOrdered.[Customer Key] = Customers.[Customer Key] AND    -- make sure to match the customer keys of those that made orders
      CustsOrdered.[Customer Key] = Orders.[Customer Key];

Proposition #5

Find all distinct suppliers of active transactions (of purchases/orders that were being shipped around) from 1/1/2013 to 1/10/2013. This is to see which suppliers were responsible for supplying for active transactions in a specific date range. (use Fact.Transaction and Fact.Movement)

In [ ]:
USE WideWorldImportersDW;

WITH PurchaseId as (    -- find all purchase order IDs that have both been in transactions and shipped around
    SELECT [WWI Purchase Order ID]
    FROM Fact.[Transaction]

    INTERSECT

    SELECT [WWI Purchase Order ID]
    FROM Fact.[Movement]
)

SELECT DISTINCT Supplier.[Supplier Key]
FROM PurchaseId, Dimension.Supplier as Supplier, Fact.[Transaction] as Trans, Fact.Movement as Move
WHERE '2013-01-01' <= Trans.[Date Key] AND  -- make sure that the only active transactions left have been shipped around from 1/1/2013 to 1/10/2013
      Trans.[Date Key] <= '2013-01-10' AND 
      '2013-01-01' <= Move.[Date Key] AND
      Move.[Date Key] <= '2013-01-10'

Proposition #6

Find all postal codes (in descending order) that the customer base of WWI doesn't share with the suppliers of WWI. This is to see where customers live outside the locations of suppliers. (use Dimension.Customer and Dimension.Supplier)

In [ ]:
USE WideWorldImportersDW;

WITH OtherZips as(  -- get all postal codes that customers that don't share with WWI suppliers
    SELECT [Postal Code] as 'Customer-Only Postal Code'
    FROM Dimension.Customer

    EXCEPT

    SELECT [Postal Code]
    FROM Dimension.Supplier
)

-- return customer-only postal codes
SELECT [Customer-Only Postal Code]
FROM OtherZips
ORDER BY [Customer-Only Postal Code] desc;

Proposition #7

Find the top 5 items with the highest quantities that can be found in both purchases and on-hand stocks/goods. This is to see what items are both highest in demand ever, followed by what's in stock. Since we got no results back, that means that there are no items that have both ever been in demand and are in stock (use Fact.Purchase and Fact.Stock Holding)

In [ ]:
USE WideWorldImportersDW;

WITH ItemKeys as(   -- get all item keys/IDs found in both purchases and on-hand stock
    SELECT [Stock Item Key]
    FROM Fact.[Purchase]

    INTERSECT

    SELECT [Stock Item Key]
    FROM Fact.[Stock Holding]
)

SELECT TOP 5 ItemKeys.[Stock Item Key],         -- get the top 5 items with the highestt quantities
             Purchases.[Ordered Quantity] as 'Purchased Quantity', 
             Stocks.[Quantity On Hand] as 'Stocks on Hand Quantity'
FROM ItemKeys, Fact.Purchase as Purchases, Fact.[Stock Holding] as Stocks
ORDER BY Purchases.[Ordered Quantity] desc,     -- sort by purchases' quantity first, followed by stocks on hand and itemkeys in descending order
         Stocks.[Quantity On Hand] desc, 
         ItemKeys.[Stock Item Key] desc;

Proposition #8

Find the end dates, starting with the earliest and going up from there, of how long all employees and suppliers will keep working with WWI. This is to see the earliest dates of when the WWI will start losing their current suppliers or employees. (use Dimension.Employee and <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Dimension.Supplier)</span>

In [ ]:
USE WideWorldImportersDW;

WITH EndDates as(           -- get all end dates of employees and suppliers involvement with WWI
    SELECT [Valid To]
    FROM Dimension.[Employee]

    UNION

    SELECT [Valid To]
    FROM Dimension.Supplier
)

SELECT *
FROM EndDates
ORDER BY [Valid To] asc;    -- sort end dates in ascending order

Proposition #9

Find all <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">transaction types of each transaction</span><span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">. This is to see how many transactions have been made with each registered type. (use Dimension.Transaction Type and Fact.Transaction)</span>

In [ ]:
USE WideWorldImportersDW;

WITH TransKeys as (     -- get all rows of common transaction type keys between all transactions and transaction types
    SELECT [Transaction Type Key]
    FROM Fact.[Transaction]

    INTERSECT

    SELECT [Transaction Type Key]
    FROM Dimension.[Transaction Type]
)

-- return all rows that share a transaction type key, with each row representing a transaction, in ascending order
SELECT TransType.[Transaction Type]
FROM TransKeys, Dimension.[Transaction Type] as TransType
ORDER BY TransType.[Transaction Type Key]

Proposition #10

Find all cities that don't start with the letters A, B, or C, sorted in ascending order. This is to see what other registered cities are in the WWI database. <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">(use Dimension.City)</span>

In [ ]:
USE WideWorldImportersDW;

WITH Non_ABC_Cities as(     -- get all cities that don't start with an A, B, or C in the WWI database
    ((SELECT City
    FROM Dimension.City as AllCities

    EXCEPT

    -- eliminate all cities that start with an A
    SELECT City
    FROM Dimension.City as A_Cities
    WHERE City LIKE N'A%')



    EXCEPT

    -- eliminate all cities that start with a B
    SELECT City
    FROM Dimension.City as B_Cities
    WHERE City LIKE N'B%')



    EXCEPT

    -- eliminate all cities that start with a C
    SELECT City
    FROM Dimension.City as C_Cities
    WHERE City LIKE N'C%'
)

SELECT *
FROM Non_ABC_Cities
ORDER BY City;